In [ ]:
# @title 1. Install Dependencies
!pip install -q dm-haiku chex orbax-checkpoint jaxtyping optax chess scipy websockets


In [ ]:
# @title 2. Get Code & Weights
import os
from google.colab import drive

# 1. Clone the code from GitHub
GITHUB_REPO = "https://github.com/your-username/searchless_chess.git" # <-- UPDATE THIS URL
if not os.path.exists('/content/searchless_chess'):
    !git clone {GITHUB_REPO} /content/searchless_chess
else:
    !cd /content/searchless_chess && git pull

# 2. Mount Drive just to access the heavy checkpoints
drive.mount('/content/drive')


In [ ]:
# @title 3. Imports
import sys
sys.path.append('/content/')  # So Python finds the cloned 'searchless_chess' folder

import os
import chess
import chess.svg
from jax import random as jrandom
import numpy as np


In [ ]:
# @title 4. Import Searchless Chess Modules
from searchless_chess.src import tokenizer
from searchless_chess.src import training_utils
from searchless_chess.src import transformer
from searchless_chess.src import utils
from searchless_chess.src.engines import engine
from searchless_chess.src.engines import neural_engines


In [ ]:
# @title 5. Create the Predictor (270M Config)

policy = 'action_value'
num_return_buckets = 128

match policy:
  case 'action_value':
    output_size = num_return_buckets
  case 'behavioral_cloning':
    output_size = utils.NUM_ACTIONS
  case 'state_value':
    output_size = num_return_buckets
  case _:
    raise ValueError(f'Unknown policy: {policy}')

predictor_config = transformer.TransformerConfig(
    vocab_size=utils.NUM_ACTIONS,
    output_size=output_size,
    pos_encodings=transformer.PositionalEncodings.LEARNED,
    max_sequence_length=tokenizer.SEQUENCE_LENGTH + 2,
    num_heads=8,
    num_layers=16,
    embedding_dim=1024,
    apply_post_ln=True,
    apply_qk_layernorm=False,
    use_causal_mask=False,
)

predictor = transformer.build_transformer_predictor(config=predictor_config)


In [ ]:
# @title 6. Load the 270M Checkpoint from Drive

# We load the weights from Drive since they are too big for GitHub
checkpoint_dir = '/content/drive/MyDrive/checkpoints/270M/' # <-- UPDATE THIS IF NEEDED

dummy_params = predictor.initial_params(
    rng=jrandom.PRNGKey(0),
    targets=np.zeros((1, 1), dtype=np.uint32),
)
params = training_utils.load_parameters(
    checkpoint_dir=checkpoint_dir,
    params=dummy_params,
    use_ema_params=True,
    step=-1,
)

# Cast weights to bfloat16 for faster GPU inference
import jax
params = jax.tree_util.tree_map(lambda x: x.astype(jax.numpy.bfloat16), params)
print('✅ 270M model loaded on', jax.devices()[0])


In [ ]:
# @title 7. Create the Engine

predict_fn = neural_engines.wrap_predict_fn(predictor, params, batch_size=1)
_, return_buckets_values = utils.get_uniform_buckets_edges_values(
    num_return_buckets
)

neural_engine = neural_engines.ENGINE_FROM_POLICY[policy](
    return_buckets_values=return_buckets_values,
    predict_fn=predict_fn,
    temperature=0.005,
)
print('✅ Engine ready!')


In [ ]:
# @title 8. Quick Sanity Check — Play a Move
board = chess.Board()
best_move = neural_engine.play(board)
print(f'Best move from starting position: {best_move}')


In [ ]:
# @title 9. Start WebSocket Server + Tunnel to Chrome Extension
import asyncio
import websockets
import json
import chess
import chess.polyglot
import logging
import threading
import time

# Suppress connection noise
logging.getLogger('websockets').setLevel(logging.ERROR)

# === 1. LOAD THE OPENING BOOK ===
try:
    # Since we cloned the code, the opening book is directly inside the cloned repo
    reader = chess.polyglot.open_reader('/content/searchless_chess/src/opening_book.bin')
    print('📚 Opening Book loaded!')
except FileNotFoundError:
    print('⚠️ No opening_book.bin found.')
    reader = None

async def handle_bridge(websocket):
    print('🌐 Extension connected!')
    try:
        async for message in websocket:
            data = json.loads(message)
            req_type = data.get('type', 'play')
            try:
                if req_type == 'reset':
                    print('♻️ Reset triggered.')
                    continue

                incoming_fen = data.get('fen', '')
                board = chess.Board(incoming_fen)

                if board.is_game_over() or len(list(board.legal_moves)) == 0:
                    continue

                if req_type == 'play':
                    t0 = time.perf_counter()

                    # Opening book check
                    if reader:
                        try:
                            book_entry = reader.find(board)
                            await websocket.send(json.dumps({'best_move': book_entry.move.uci()}))
                            t_book = time.perf_counter()
                            print(f'📚 BOOK HIT! {(t_book - t0)*1000:.2f} ms')
                            continue
                        except IndexError:
                            pass

                    t1 = time.perf_counter()
                    best_move = neural_engine.play(board)
                    t2 = time.perf_counter()
                    await websocket.send(json.dumps({'best_move': best_move.uci()}))
                    t3 = time.perf_counter()

                    print(f'🧠 Move: {best_move.uci()} | GPU: {(t2-t1)*1000:.0f}ms | Total: {(t3-t0)*1000:.0f}ms')

            except ValueError:
                pass
    except websockets.exceptions.ConnectionClosed:
        print('🔌 Extension disconnected.')

async def start_server():
    async with websockets.serve(handle_bridge, '0.0.0.0', 8000):
        print('🚀 WebSocket server running on port 8000')
        await asyncio.Future()

def run_server_in_thread():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_server())

threading.Thread(target=run_server_in_thread, daemon=True).start()

# === Setup localtunnel to expose the server ===
import subprocess, urllib.request

# Install localtunnel
subprocess.run(['npm', 'install', '-g', 'localtunnel'], capture_output=True)

# Get public IP (needed if localtunnel shows a warning page)
try:
    my_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
    print(f'\n⚠️ If localtunnel shows a warning page, enter this IP as the password: {my_ip}')
except:
    pass

# Start tunnel in background and capture URL
!nohup lt --port 8000 > /tmp/tunnel.log 2>&1 &
time.sleep(3)

with open('/tmp/tunnel.log', 'r') as f:
    log = f.read()
    if 'your url is: ' in log:
        url = log.split('your url is: ')[-1].strip()
        wss_url = url.replace('https://', 'wss://')
        print(f'\n✅ TUNNEL ACTIVE!')
        print(f'🔗 Update your Chrome extension to connect to:')
        print(f'   {wss_url}')
    else:
        print('Tunnel output:', log)
        print('Try re-running this cell.')
